In [ ]:
# 1. Imports
import sys
import os
from pathlib import Path

def get_project_root():
    if "__file__" not in globals():
        cwd = Path(os.getcwd()).resolve()
        for parent in [cwd] + list(cwd.parents):
            if (parent / "src").exists():
                return parent
        return cwd
    return Path(__file__).resolve().parents[1]

ROOT = get_project_root()
sys.path.insert(0, str(ROOT))
print("Project root added to sys.path:", ROOT)

import numpy as np
import pandas as pd

# Embedded datasets
from src.likelihoods.data.planck_2015 import PLANCK_2015
from src.likelihoods.data.planck_2018_recon import PLANCK_2018_RECON
from src.likelihoods.data.desi_bao_dr1 import DESI_BAO_DR1
from src.likelihoods.data.cosmic_chronometers import COSMIC_CHRONOMETERS
from src.likelihoods.data.pantheon_plus_full import PANTHEON_PLUS_FULL

# Likelihoods
from src.likelihoods.planck_shoes_joint import PlanckSH0ESJointLikelihood
from src.likelihoods.desi_bao import DESIBAO
from src.likelihoods.cosmic_chronometers import CosmicChronometers
from src.likelihoods.joint_likelihood import JointLikelihood

# MCMC
from src.physics.mcmc_joint_pipeline import JointMCMCPipeline

# Modernized figure pipeline
from src.pipeline.paper_figures_pipeline import PaperFiguresPipeline
from src.analysis.latex_constraints import constraints_to_latex


In [ ]:
# 2. Build likelihoods from embedded data

def build_joint_likelihood(planck_dict):
    planck_like = PlanckSH0ESJointLikelihood(planck_dict)
    bao_like = DESIBAO(DESI_BAO_DR1)
    cc_like = CosmicChronometers(COSMIC_CHRONOMETERS)
    return JointLikelihood(planck_like, bao_like, cc_like)

joint_2015 = build_joint_likelihood(PLANCK_2015)
joint_2018 = build_joint_likelihood(PLANCK_2018_RECON)


In [ ]:
# 3. Define S.T.A.R. Model

H_expr = "H0*sqrt(Ωm*(1+z)**3 + ΩΛ + a*z + b*z**2)"
param_names = ["H0", "Ωm", "ΩΛ", "a", "b"]

priors = {
    "H0": (70, 2),
    "Ωm": (0.3, 0.05),
    "ΩΛ": (0.7, 0.05),
    "a": (0, 0.02),
    "b": (0, 0.02),
}

proposal_widths = {
    "H0": 0.1,
    "Ωm": 0.01,
    "ΩΛ": 0.01,
    "a": 0.002,
    "b": 0.002,
}


In [ ]:
# 4. Run MCMC for both Planck datasets

pipeline_2015 = JointMCMCPipeline(H_expr, param_names, priors, proposal_widths, joint_2015)
chain_2015 = pipeline_2015.run(theta0=[70, 0.3, 0.7, 0, 0], nsteps=8000)


In [ ]:
# 5. Generate paper figures for both runs

fig_pipeline_2015 = PaperFiguresPipeline(
    chain_2015,
    param_names,
    H_expr,
    data_paths={"sn": PANTHEON_PLUS_FULL}
)
constraints_2015 = fig_pipeline_2015.run("paper_figures_planck2015")


constraints_2015 = fig_pipeline_2015.run("paper_figures_planck2015")


fig_pipeline_2018 = PaperFiguresPipeline(
    chain_2018,
    param_names,
    H_expr,
    data_paths={"sn": PANTHEON_PLUS_FULL}
)

constraints_2018 = fig_pipeline_2018.run("paper_figures_planck2018")

constraints_2015, constraints_2018

